In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import sys
import os

# rapids-singlecell
import cupy as cp
import rapids_singlecell as rsc

In [2]:
# figure parameter setting
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

%matplotlib inline
mpl.rcParams.update(mpl.rcParamsDefault) #Reset rcParams to default

# Editable text and proper LaTeX fonts in illustrator
# matplotlib.rcParams['ps.useafm'] = True
# Editable fonts. 42 is the magic number
mpl.rcParams['pdf.fonttype'] = 42
sns.set(style='whitegrid', context='paper')
# Set default DPI for saved figures
mpl.rcParams['savefig.dpi'] = 600

In [3]:
import logging
# Suppress INFO-level logs for the entire logger
logging.getLogger().setLevel(logging.WARNING)

In [4]:
# define the figure path
figpath = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/zebrahub-multiome-analysis/figures/chrom_velo/"
os.makedirs(figpath, exist_ok=True)
sc.settings.figdir = figpath

In [5]:
# # define the filepath to save the subsetted adata objects
# filepath = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/adata_peak_clusters_subset/"
# filepath

'/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/adata_peak_clusters_subset/'

In [5]:
# import the peaks-by-celltype&timepoint pseudobulk object
adata_peaks = sc.read_h5ad("/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/annotated_data/objects_v2/peaks_by_pb_leiden_0.4_merged_annotated_filtered.h5ad")
adata_peaks

AnnData object with n_obs × n_vars = 640830 × 190
    obs: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'accessibility_neural_optic', 'accessibility_endoderm', 'accessibility_enteric_neurons', 'accessibility_notochord', 'accessibility_pharyngeal_arches', 'accessibility_floor_plate', 'accessibility_epidermis', 'accessibility_pronephros', 'accessibility_neural_floor_plate', 'accessibility_lateral_plate_mesoderm', 'accessibility_midbrain_hindbrain_boundary', 'accessibility_neural', 'accessibility_neurons', 'accessibility_neural_posterior', 'accessibility_neural_crest', 'accessibility_PSM', 'accessibility_fast_muscle', 'accessibility_endocrine_pancreas', 'accessibility_heart_myocardium', 'accessibility_neural_telencephalon', 'accessibility_optic_cup', 'accessibility_primordial_germ_cells', 'accessibility_differentiating_neurons', 'accessibility_muscle', 'accessibility_tail_bud', 'accessibility_hindbrain', 'accessibility_somites', 'accessibility_spinal_cord',

In [6]:
# import the cicero output (long format)
def load_coaccessibility_matrix_longformat(matrix_path, 
                                         peak1_col='Peak1', 
                                         peak2_col='Peak2', 
                                         coaccess_col='coaccess',
                                         threshold=0.1, 
                                         symmetric=True,
                                         add_self_loops=True):
    """
    Load co-accessibility matrix from long format (Peak1, Peak2, coaccess) to peak-by-peak matrix.
    
    Parameters:
    -----------
    matrix_path : str
        Path to the co-accessibility file
    peak1_col : str
        Name of the first peak column
    peak2_col : str
        Name of the second peak column  
    coaccess_col : str
        Name of the co-accessibility score column
    threshold : float
        Minimum co-accessibility score to keep (default: 0.1)
    symmetric : bool
        Whether to make the matrix symmetric (default: True)
    add_self_loops : bool
        Whether to add self-loops with score 1.0 (default: True)
    
    Returns:
    --------
    coaccess_matrix : numpy.ndarray
        Peak-by-peak co-accessibility matrix
    peak_names : list
        List of peak names corresponding to matrix rows/columns
    """
    print(f"Loading co-accessibility matrix from {matrix_path}")
    
    # Load the data
    df = pd.read_csv(matrix_path)
    
    # Get column names (handle case where columns might be named differently)
    if peak1_col not in df.columns:
        # Try to find the peak columns automatically
        cols = df.columns.tolist()
        if len(cols) >= 3:
            peak1_col, peak2_col, coaccess_col = cols[:3]
            print(f"Auto-detected columns: {peak1_col}, {peak2_col}, {coaccess_col}")
        else:
            raise ValueError(f"Could not find {peak1_col} column in {df.columns}")
    
    print(f"Using columns: {peak1_col}, {peak2_col}, {coaccess_col}")
    
    # Filter by threshold
    df_filtered = df[df[coaccess_col] >= threshold].copy()
    print(f"Filtered from {len(df)} to {len(df_filtered)} pairs (threshold >= {threshold})")
    
    # Get unique peaks
    all_peaks = sorted(list(set(df_filtered[peak1_col].tolist() + df_filtered[peak2_col].tolist())))
    n_peaks = len(all_peaks)
    peak_to_idx = {peak: idx for idx, peak in enumerate(all_peaks)}
    
    print(f"Found {n_peaks} unique peaks")
    
    # Create the matrix
    coaccess_matrix = np.zeros((n_peaks, n_peaks))
    
    # Fill the matrix
    for _, row in df_filtered.iterrows():
        peak1_idx = peak_to_idx[row[peak1_col]]
        peak2_idx = peak_to_idx[row[peak2_col]]
        score = row[coaccess_col]
        
        # Add the score
        coaccess_matrix[peak1_idx, peak2_idx] = score
        
        # Make symmetric if requested
        if symmetric and peak1_idx != peak2_idx:
            coaccess_matrix[peak2_idx, peak1_idx] = score
    
    # Add self-loops if requested
    if add_self_loops:
        np.fill_diagonal(coaccess_matrix, 1.0)
        print("Added self-loops with score 1.0")
    
    print(f"Co-accessibility matrix shape: {coaccess_matrix.shape}")
    print(f"Non-zero entries: {np.count_nonzero(coaccess_matrix)}")
    print(f"Sparsity: {1 - np.count_nonzero(coaccess_matrix) / coaccess_matrix.size:.3f}")
    
    return coaccess_matrix, all_peaks

In [12]:
# load the co-accessibility matrix
cicero_output_path = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/cicero_integrated_ATAC/02_integrated_ATAC_cicero_connections_peaks_integrated.csv"
coaccess_matrix, peak_names_coaccess = load_coaccessibility_matrix_longformat(
    cicero_output_path, 
    peak1_col='Peak1', 
    peak2_col='Peak2', 
    coaccess_col='coaccess',
    threshold=0,  # Changed from 0.8 to 0.1 for more connections
    symmetric=True,
    add_self_loops=True
)

Loading co-accessibility matrix from /hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/13_peak_umap_analysis/cicero_integrated_ATAC/02_integrated_ATAC_cicero_connections_peaks_integrated.csv
Using columns: Peak1, Peak2, coaccess
Filtered from 5405656 to 3697174 pairs (threshold >= 0)
Found 31613 unique peaks
Added self-loops with score 1.0
Co-accessibility matrix shape: (31613, 31613)
Non-zero entries: 3151661
Sparsity: 0.997


In [20]:
test = pd.read_csv(cicero_output_path, index_col=0)
test.head()

,Peak1,Peak2,coaccess
2,1-14250900-14251605,1-14251729-14252087,0.100097
3,1-14250900-14251605,1-14252873-14253611,0.001212
4,1-14250900-14251605,1-14256357-14256643,0.001017
5,1-14250900-14251605,1-14257183-14257984,-0.001604
6,1-14250900-14251605,1-14259756-14259970,-0.001339


In [17]:
len(test.Peak1.unique())

31613

In [ ]:
for chrom in np.arange(0,26):
    # check if the 

In [14]:
# test_matrix, peak_names_test = load_coaccessibility_matrix_longformat(
#     "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/02_cicero_processed/TDR118reseq_cicero/02_TDR118reseq_cicero_connections_peaks_merged_peaks.csv",
#     peak1_col='Peak1', 
#     peak2_col='Peak2', 
#     coaccess_col='coaccess',
#     threshold=0,  # Changed from 0.8 to 0.1 for more connections
#     symmetric=True,
#     add_self_loops=True)

Loading co-accessibility matrix from /hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/02_cicero_processed/TDR118reseq_cicero/02_TDR118reseq_cicero_connections_peaks_merged_peaks.csv
Using columns: Peak1, Peak2, coaccess
Filtered from 100609902 to 64770914 pairs (threshold >= 0)
Found 404660 unique peaks


MemoryError: Unable to allocate 1.19 TiB for an array with shape (404660, 404660) and data type float64

In [18]:
cicero_path = "/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/02_cicero_processed/TDR118reseq_cicero/02_TDR118reseq_cicero_connections_peaks_merged_peaks.csv"
test= pd.read_csv(cicero_path, index_col=0)
test.head()

,Peak1,Peak2,coaccess
1,1-10000286-10000789,1-9753075-9753596,0.000000
2,1-10000286-10000789,1-9759496-9760011,-0.005967
3,1-10000286-10000789,1-9764309-9764532,-0.005358
4,1-10000286-10000789,1-9768426-9768767,0.002992
5,1-10000286-10000789,1-9770155-9770487,-0.069031


In [19]:
len(test.Peak1.unique())

404660

In [ ]:
# sanity check for the peaks from adata_peaks and coaccess_matrix

In [11]:
for peak in peak_names_coaccess:
    if peak in adata_peaks.obs_names:
        pass
    else:
        print(peak)

10-45419551-45420917
